# Simulating Population Groups Distributed Randomly in Space

For conducting single-value inference, the `segregation` package offers several techniques for generating random population distributions that respect the characteristics of an input dataset. This notebook walks through the assumptions and outputs of each approach using the Sacramento demonstration dataset bundled with `libpysal`.

## TL;DR

- evenness includes group-level variation
- systematic includes unit-level variation
- individual permutation includes neither

In [ ]:
%load_ext watermark
%watermark -a 'eli knaap' -v -d -u -p segregation,geopandas,libpysal

In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from libpysal.examples import load_example
from segregation.singlegroup import Gini
from segregation.multigroup import MultiInfoTheory
from segregation.inference import (
    SingleValueTest,
    simulate_evenness,
    simulate_person_permutation,
    simulate_systematic_randomization,
    simulate_null,
)

In [ ]:
sacramento = gpd.read_file(load_example("Sacramento1").get_path("sacramentot2.shp"))
sacramento = sacramento.to_crs(sacramento.estimate_utm_crs())

We focus on the Black population (`BLACK`) relative to the total population (`TOT_POP`) of each tract.

In [ ]:
sacramento["BLACK"].sum()

In [ ]:
sacramento["TOT_POP"].sum()

In [ ]:
sacramento["BLACK"].sum() / sacramento["TOT_POP"].sum()

In [ ]:
gini = Gini(sacramento, group_pop_var="BLACK", total_pop_var="TOT_POP")

## Evenness

Evenness takes draws from the population of each unit, with the probability of choosing the focal group equal to its regional share (locations drawing from distributions of population groups).

In [ ]:
# the region-wide share of the Black population
sacramento["BLACK"].sum() / sacramento["TOT_POP"].sum()

In [ ]:
sacramento[["TOT_POP"]].reset_index(drop=True).head()

For the first tract, one draw is taken per resident; on each draw the probability that the resident belongs to the focal group equals the region-wide share computed above.

In [ ]:
evenness = simulate_evenness(sacramento, group="BLACK", total="TOT_POP")

In [ ]:
evenness

In [ ]:
evenness.plot("BLACK", scheme="quantiles")

In [ ]:
evenness["BLACK"].sum()

In [ ]:
evenness["BLACK"].sum() == sacramento["BLACK"].sum()

In [ ]:
evenness["BLACK"].sum() / evenness["TOT_POP"].sum()

In [ ]:
sacramento["BLACK"].sum() / sacramento["TOT_POP"].sum()

In [ ]:
evenness["BLACK"].sum() / evenness["TOT_POP"].sum() == sacramento["BLACK"].sum() / sacramento["TOT_POP"].sum()

In [ ]:
evenness["TOT_POP"].sum() == sacramento["TOT_POP"].sum()

In [ ]:
np.array_equal(evenness["TOT_POP"].values, sacramento["TOT_POP"].values)

We haven't changed the total population in each unit or in the region, but we have changed the number of people in the focal group marginally.

## Systematic Randomization

The systematic approach takes draws from the regional population of each group, with the probability of choosing a geographic unit equal to the share of the region's population that currently lives there (people drawing from a distribution of locations).

In [ ]:
sacramento["BLACK"].sum()

In [ ]:
(sacramento["TOT_POP"] / sacramento["TOT_POP"].sum()).reset_index(drop=True).head()

For each member of the focal group, the probability of being placed in a given tract equals that tract's share of the total population (shown above for the first few tracts).

In [ ]:
systematic = simulate_systematic_randomization(
    sacramento, group="BLACK", total="TOT_POP"
)

In [ ]:
systematic

In [ ]:
systematic["BLACK"].sum()

In [ ]:
systematic["BLACK"].sum() == sacramento["BLACK"].sum()

In [ ]:
systematic["BLACK"].sum() / systematic["TOT_POP"].sum() == sacramento["BLACK"].sum() / sacramento["TOT_POP"].sum()

In [ ]:
np.array_equal(systematic["TOT_POP"].values, sacramento["TOT_POP"].values)

We haven't changed the total number of people in each group, but we have changed the total number of people in each unit.

### Individual-level Permutation

Individual-level permutation doesn't take draws from a probability distribution, but instead randomizes which unit each person lives in.

In [ ]:
permutation = simulate_person_permutation(
    sacramento, group="BLACK", total="TOT_POP"
)

In [ ]:
permutation["BLACK"].sum()

In [ ]:
permutation["BLACK"].sum() == sacramento["BLACK"].sum()

In [ ]:
permutation["BLACK"].sum() / permutation["TOT_POP"].sum() == sacramento["BLACK"].sum() / sacramento["TOT_POP"].sum()

We haven't changed the total number of people in any group, or the total population in any unit; we've only randomized which unit each person lives in.

## Simulating Null Distributions

`simulate_null` generates a series of simulated segregation statistics (in parallel) using the randomization functions described above. Those simulated values can then serve as a reference distribution to test the hypothesis of "no segregation".

In [ ]:
groups = ["BLACK", "WHITE", "ASIAN", "HISP"]

In [ ]:
G = Gini(sacramento, group_pop_var="BLACK", total_pop_var="TOT_POP")

In [ ]:
H = MultiInfoTheory(sacramento, groups=groups)

In [ ]:
G.statistic

In [ ]:
H.statistic

### Single Group

In [ ]:
G_even = simulate_null(seg_class=G, sim_func=simulate_evenness)

In [ ]:
G_systematic = simulate_null(seg_class=G, sim_func=simulate_systematic_randomization)

In [ ]:
G_permuted = simulate_null(seg_class=G, sim_func=simulate_person_permutation)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for series, label in [
    (G_permuted, "permuted"),
    (G_systematic, "systematic"),
    (G_even, "evenness"),
]:
    series.name = label
    series.plot(kind="kde", ax=ax, legend=True)

### Multi Group

In [ ]:
H_even = simulate_null(seg_class=H, sim_func=simulate_evenness)

In [ ]:
H_systematic = simulate_null(seg_class=H, sim_func=simulate_systematic_randomization)

In [ ]:
H_permuted = simulate_null(seg_class=H, sim_func=simulate_person_permutation)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for series, label in [
    (H_permuted, "permuted"),
    (H_systematic, "systematic"),
    (H_even, "evenness"),
]:
    series.name = label
    series.plot(kind="kde", ax=ax, legend=True)

Despite their different methods, all three approaches simulate similar distributions, but they differ with respect to *how* and in which dimensions the randomization occurs. As with [Boisso et al.](http://dx.doi.org/10.1016/0304-4076(94)90082-5), the distribution is not centered on 0. In other cases, such as when minority populations are small or highly unbalanced among multiple groups, it is possible that the different randomization methods could diverge to simulate different distributions.